# Supervisor Agent 기반 멀티 에이전트 워크플로우 (Human-in-the-Loop 포함)

본 노트북은 **LangGraph**를 활용하여 사용자의 복합적인 요청을 처리하는 **Supervisor 패턴 기반의 멀티 에이전트 시스템**을 구현한 예제입니다. 전체 파이프라인은 **계획 수립 → 정보 수집 → 데이터 분석 → 초안 작성 → 품질 검토 → 사용자 최종 승인(Human-in-the-Loop)**의 6단계로 구성되어 있으며, 조건부 라우팅(Conditional Routing)을 통해 자체 피드백 루프와 재작성 횟수 제한 로직을 제어합니다.

---

### 전체 아키텍처 및 에이전트 역할 안내
1. **Request/Planner Agent (`request_planner`)**:
   - 사용자의 원본 요청(`input`)을 분석하여 순차적인 세부 작업 계획(`steps`)과 정보 수집을 위한 검색 질의 목록(`search_queries`)을 구조화된 형태(Pydantic 모델)로 생성합니다.
2. **Retrieval Agent (`retrieval`)**:
   - 수립된 작업 계획과 검색 질의 목록을 바탕으로 관련성 높은 핵심 배경지식과 기초 자료(`retrieved_docs`)를 수집하고 정리합니다.
3. **Analysis Tool Agent (`analysis`)**:
   - 수집된 정보를 다각도로 분석하여 보고서 작성에 필요한 핵심 인사이트, 트렌드, 시사점(`analysis`)을 도출합니다.
4. **Revision Counter Node (`increment_revision`)**:
   - 초안 작성(또는 재작성)에 들어가기 직전에 호출되어 현재 작성 회차(`revision_count`)를 1씩 증가시킵니다. 무한 루프를 방지하는 기준값으로 활용됩니다.
5. **Writer Agent (`writer`)**:
   - 원본 요청, 작업 계획, 분석 결과를 종합하여 체계적인 보고서 초안(`draft`)을 작성합니다. 만약 이전 단계에서 검토자(Reviewer)나 사용자(Human)로부터 반려 피드백(`review_feedback`)을 받은 경우, 이를 반영하여 초안을 개선합니다.
6. **Reviewer Agent (`reviewer`)**:
   - 작성된 초안이 사용자의 요구사항을 충족하는지 엄격하게 평가합니다. 승인 여부(`review_approved`)와 구체적인 개선 피드백(`review_feedback`)을 JSON 스키마 기반으로 출력합니다.
7. **Human Approval Node (`human_approval`)**:
   - LangGraph의 `interrupt()` 기능을 사용하여 그래프 실행을 일시 중지하고 사용자의 최종 확인을 대기합니다. 사용자가 승인(`approve`)하면 워크플로우를 종료하고, 수정 사항을 입력하여 반려하면 다시 Writer Agent로 되돌려 재작성을 수행합니다.


## 1. 필수 패키지 설치

워크플로우 구현에 필요한 핵심 라이브러리를 설치합니다.
- **`langchain`**: LLM 기반 애플리케이션 개발을 위한 핵심 프레임워크로, 메시지 객체(`SystemMessage`, `HumanMessage`) 등을 제공합니다.
- **`langchain-openai`**: OpenAI 호환 API(NVIDIA NIM, LM Studio, vLLM 등 포함)와 통신하기 위한 클라이언트(`ChatOpenAI`)를 제공합니다.
- **`langgraph`**: 상태 기반(Stateful) 멀티 에이전트 그래프, 조건부 분기, 체크포인터(Checkpointer), Human-in-the-Loop(`interrupt`, `Command`)를 구현하기 위한 핵심 라이브러리입니다.
- **`pydantic`**: LLM의 출력을 정형화된 데이터 구조(Structured Output)로 강제하고 데이터 유효성을 검증하기 위해 사용합니다.


In [ ]:
# 필요한 패키지를 조용히(-q 옵션) 설치합니다.
%pip install -q langchain langchain-openai langgraph pydantic


## 2. 모듈 임포트 및 입출력 인코딩 설정

그래프 구성, 타입 힌팅, 구조화된 출력 검증, 시각화 등에 필요한 파이썬 모듈들을 임포트합니다.
또한, Windows 환경이나 일부 Jupyter 커널 환경에서 한글 출력 시 발생할 수 있는 인코딩 오류(`UnicodeEncodeError`)를 방지하기 위해 표준 출력(`stdout`)과 표준 에러(`stderr`) 스트림을 **UTF-8**로 재설정합니다.


In [ ]:
# 타입 힌팅 및 데이터 구조 정의를 위한 표준/확장 모듈을 임포트합니다.
from typing import List
from typing_extensions import Literal, TypedDict

# LLM의 구조화된 출력(Structured Output) 스키마 정의를 위한 Pydantic 모듈을 임포트합니다.
from pydantic import BaseModel, Field

# LLM에게 전달할 시스템 프롬프트와 사용자 메시지 클래스 및 OpenAI 호환 클라이언트를 임포트합니다.
from langchain.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

# LangGraph의 핵심 구성 요소(상태 그래프, 시작/종료 노드, 메모리 체크포인터, 인터럽트 제어 도구)를 임포트합니다.
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# 그래프 시각화 및 파일 경로/시스템 제어를 위한 유틸리티 모듈을 임포트합니다.
from IPython.display import Image, display
from pathlib import Path
import json
import re
import sys

# 현재 시스템의 표준 출력 인코딩이 UTF-8이 아닌 경우, 한글 깨짐 방지를 위해 UTF-8로 재설정합니다.
# errors="replace" 옵션을 통해 인코딩 불가능한 문자가 있더라도 예외로 중단되지 않도록 처리합니다.
if sys.stdout.encoding and sys.stdout.encoding.lower().replace("-", "") != "utf8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")


## 3. 구조화된 출력(Structured Output)을 위한 Pydantic 스키마 정의

에이전트 간에 데이터를 안정적으로 주고받기 위해 LLM의 응답 형식을 Pydantic `BaseModel`로 엄격하게 정의합니다.

- **`Plan` 클래스**:
  - `Request/Planner Agent`가 생성하는 출력 구조입니다.
  - 오픈 웨이트 모델(예: `gpt-oss-20b` 등)에서 중첩된 객체 리스트(`List[PlanStep]`)를 생성할 때 발생할 수 있는 JSON 파싱 불안정성을 예방하기 위해, 평탄화된 문자열 리스트(`List[str]`) 구조를 채택하였습니다.
  - `Field`의 `min_length`, `max_length` 제약을 통해 단계 수(2~6개)와 검색어 수(2~5개)를 강제합니다.
- **`ReviewResult` 클래스**:
  - `Reviewer Agent`가 초안을 검토한 뒤 출력하는 구조입니다.
  - 구체적인 개선 의견을 담는 `feedback`(`str`)과 품질 기준 통과 여부를 나타내는 `approved`(`bool`) 필드로 구성됩니다.


In [ ]:
class Plan(BaseModel):
    """Request/Planner Agent가 생성하는 작업 계획 스키마입니다.

    주의: 중첩 객체 구조(PlanStep)는 gpt-oss 등 일부 모델에서 구조화된 출력이
    불안정할 수 있으므로, 파싱 안정성이 높은 단순한 List[str] 구조를 사용합니다.
    각 step 문자열은 "작업내용 (핵심: 키워드1, 키워드2)" 형식으로 작성하도록 유도합니다.
    """
    # 순차적 작업 단계를 저장하는 문자열 리스트입니다. (최소 2개 ~ 최대 6개)
    steps: List[str] = Field(
        description=(
            "사용자 요청을 처리하기 위한 순차적 작업 계획 (최소 2개, 최대 6개). "
            '각 단계는 "작업내용 (핵심: 키워드1, 키워드2)" 형식의 문자열입니다. '
            '예: "사용자 요청의 핵심 주제와 목적을 분석한다 (핵심: 요청 분석, 핵심 주제)"'
        ),
        min_length=2,
        max_length=6,
    )
    # Retrieval Agent가 정보 수집에 활용할 검색어 리스트입니다. (최소 2개 ~ 최대 5개)
    search_queries: List[str] = Field(
        description=(
            "Retrieval Agent가 사용할 검색 질의 목록 (최소 2개, 최대 5개). "
            "각 질의는 검색 엔진에 입력할 수 있는 명확한 한국어 문장 또는 키워드 조합입니다. "
            '예: "AI 에이전트 발전 과정"'
        ),
        min_length=2,
        max_length=5,
    )


class ReviewResult(BaseModel):
    """Reviewer Agent가 생성하는 초안 검토 결과 스키마입니다."""
    # 초안의 부족한 점이나 보완해야 할 구체적인 피드백을 저장합니다.
    feedback: str = Field(description="초안에 대한 피드백/개선점")
    # 초안이 요구사항을 충족하여 최종 사용자 승인 단계로 넘어갈 수 있는지 여부를 나타냅니다.
    approved: bool = Field(description="초안이 품질 기준을 통과했는지 여부")


## 4. LLM 클라이언트 초기화 및 스키마 바인딩

OpenAI 호환 API 엔드포인트(예: NVIDIA NIM API 또는 로컬 서빙 백엔드)에 연결하는 `ChatOpenAI` 인스턴스를 생성합니다.
- 일반 텍스트 생성을 수행하는 에이전트(Retrieval, Analysis, Writer)는 기본 `llm_client`를 그대로 사용합니다.
- 정형화된 응답이 필수적인 Planner와 Reviewer 에이전트에는 `.with_structured_output(..., method="json_schema")`를 적용하여, LLM의 출력이 자동으로 `Plan` 및 `ReviewResult` 객체로 역직렬화(Deserialization)되도록 설정합니다.


In [ ]:
# OpenAI 호환 API 서버에 접속하는 기본 LLM 클라이언트 객체를 생성합니다.
# 실행 전 api_key에 본인의 실제 API 키를 입력하거나 로컬 서버 주소를 설정하시기 바랍니다.
llm_client = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",  # API 엔드포인트 URL
    api_key="change_your_key",                      # 인증용 API Key (실제 키로 교체 필요)
    model="openai/gpt-oss-20b",                     # 사용할 모델 식별자
)

# Planner 에이전트용 LLM: 응답을 항상 Plan(Pydantic 모델) 인스턴스로 반환하도록 json_schema 방식을 강제합니다.
planner_llm = llm_client.with_structured_output(Plan, method="json_schema")

# Reviewer 에이전트용 LLM: 응답을 항상 ReviewResult(Pydantic 모델) 인스턴스로 반환하도록 설정합니다.
reviewer_llm = llm_client.with_structured_output(ReviewResult, method="json_schema")


## 5. 그래프 상태(Graph State) 정의

LangGraph의 모든 노드(에이전트)가 공유하고 업데이트하는 중앙 상태 저장소인 `State` 클래스를 `TypedDict`로 정의합니다. 각 노드는 `State`를 입력받아 자신의 작업 결과에 해당하는 키(Key)만 딕셔너리 형태로 반환함으로써 상태를 부분 업데이트(Partial Update)합니다.

| 상태 키 (Key) | 타입 (Type) | 상세 설명 |
| :--- | :--- | :--- |
| `input` | `str` | 사용자가 최초에 입력한 원본 요청 텍스트입니다. |
| `plan` | `str` (실행 시 `Plan` 객체) | Planner가 생성한 작업 계획 데이터입니다. |
| `search_queries` | `List[str]` | 정보 수집에 사용할 검색 질의 목록입니다. |
| `retrieved_docs` | `str` | Retrieval Agent가 수집 및 정리한 배경 정보 텍스트입니다. |
| `analysis` | `str` | Analysis Agent가 도출한 핵심 분석 및 인사이트 결과입니다. |
| `draft` | `str` | Writer Agent가 작성한 보고서 초안입니다. |
| `review_feedback` | `str` | Reviewer 또는 Human이 남긴 수정 지시사항(피드백)입니다. |
| `review_approved` | `bool` | Reviewer Agent의 품질 검토 통과 여부입니다. |
| `human_approved` | `bool` | 최종 사용자의 결과물 승인 여부입니다. |
| `final_output` | `str` | 최종 확정된 보고서 결과물입니다. |
| `revision_count` | `int` | Writer Agent가 작성(초안+재작성)을 수행한 누적 횟수입니다. |


In [ ]:
class State(TypedDict):
    """워크플로우 내의 모든 노드가 공유하고 갱신하는 상태(State) 스키마입니다."""
    input: str                  # 사용자의 원본 요청 질문/주제
    plan: str                   # 수립된 작업 계획 (Planner가 반환한 Plan 객체 저장)
    search_queries: List[str]   # 정보 검색을 위한 쿼리 리스트
    retrieved_docs: str         # 검색(수집)된 자료 요약본
    analysis: str               # 수집 자료를 기반으로 도출된 핵심 인사이트
    draft: str                  # 작성된 보고서 초안
    review_feedback: str        # 검토자(Reviewer) 또는 사용자(Human)의 수정 피드백
    review_approved: bool       # 검토자 에이전트의 승인 여부 (True/False)
    human_approved: bool        # 최종 사용자의 승인 여부 (True/False)
    final_output: str           # 최종 채택된 완료 보고서
    revision_count: int         # 작성 및 재작성 누적 횟수 (무한 루프 방지용 카운터)


## 6. 에이전트 및 작업 노드(Node) 정의

### 6-1. Request/Planner Agent (`request_planner_agent`)
사용자의 요청(`state['input']`)을 분석하여 전체 작업 단계(`steps`)와 검색어(`search_queries`)를 생성하는 첫 번째 노드입니다.
- 프롬프트 내에 구체적인 출력 규칙과 Few-shot 예시를 명시하여, 모델이 형식 오류 없이 고품질의 계획을 수립하도록 유도합니다.
- `planner_llm`을 호출하므로 반환값 `result`는 `Plan` 클래스의 인스턴스이며, 이를 통해 `result.search_queries` 속성에 안전하게 접근할 수 있습니다.


In [ ]:
def request_planner_agent(state: State):
    """사용자 요청을 분석하여 순차적 작업 계획과 검색 질의 목록을 생성합니다."""
    # 모델이 정해진 규칙과 형식을 엄격히 준수하도록 상세한 시스템 프롬프트를 구성합니다.
    system_prompt = (
        "당신은 요청 분석 및 계획 수립 전문가입니다.\n"
        "사용자의 요청을 분해하여 순차적으로 수행 가능한 작업 계획을 세우고, "
        "정보 검색에 사용할 검색 질의를 생성하세요.\n\n"
        "[출력 규칙]\n"
        "1. steps: 최소 2개, 최대 6개의 단계를 작성하세요.\n"
        '   각 단계는 "작업내용 (핵심: 키워드1, 키워드2)" 형식의 문자열입니다.\n'
        "   작업내용은 완결된 한국어 문장(10자 이상)으로 작성하세요.\n"
        "2. search_queries: 최소 2개, 최대 5개의 검색 질의를 작성하세요.\n"
        "   각 질의는 검색 엔진에 입력할 수 있는 명확한 한국어 문장 또는 키워드 조합이어야 합니다.\n"
        "   빈 문자열이나 의미 없는 문자(..., ???, ---)를 포함하지 마세요.\n\n"
        "[출력 예시]\n"
        "steps:\n"
        '  - "사용자 요청의 핵심 주제와 목적을 분석한다 (핵심: 요청 분석, 핵심 주제)"\n'
        '  - "관련 정보를 검색하여 수집한다 (핵심: 정보 검색, 자료 수집)"\n'
        "search_queries:\n"
        '  - "AI 에이전트 발전 과정"\n'
        '  - "AI 에이전트 향후 전망"'
    )

    # 구조화된 출력이 바인딩된 planner_llm을 호출하여 Plan 객체를 반환받습니다.
    result = planner_llm.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"사용자 요청: {state['input']}"),
        ]
    )
    
    # 생성된 계획 객체와 검색 질의 리스트를 State에 업데이트합니다.
    return {
        "plan": result,
        "search_queries": result.search_queries,
    }


### 6-2. Retrieval Agent (`retrieval_agent`)
Planner가 생성한 `search_queries`와 `plan`을 바탕으로 관련 정보를 수집하는 노드입니다.
- 본 예제에서는 외부 검색 API(예: Tavily, DuckDuckGo 등)를 직접 호출하는 대신, LLM의 사전 학습된 지식을 활용하여 각 검색 질의에 대한 핵심 정보를 체계적으로 정리하도록 구성되어 있습니다.
- 실제 운영 환경으로 확장할 경우, 이 노드 내부에 Vector DB 조회(RAG)나 웹 검색 도구(Tool)를 연동하여 최신 외부 정보를 가져오도록 손쉽게 교체할 수 있습니다.


In [ ]:
def retrieval_agent(state: State):
    """수립된 검색 질의 목록을 바탕으로 관련 배경지식과 핵심 정보를 수집 및 정리합니다."""
    # State에서 검색 질의 목록을 가져와 가독성 좋은 불릿 리스트 문자열로 변환합니다.
    queries = state.get("search_queries", [])
    queries_text = "\n".join(f"  - {q}" for q in queries)
    
    # 기본 LLM 클라이언트를 호출하여 각 검색 질의에 부합하는 상세 정보를 생성합니다.
    msg = llm_client.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 정보 검색 에이전트입니다. "
                    "주어진 검색 질의들에 대해 관련성 높은 정보를 수집하여 정리하세요. "
                    "실제 검색 대신, 당신의 지식을 기반으로 각 질의에 대한 핵심 정보를 제공하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"작업 계획:\n{state['plan']}\n\n"
                    f"검색 질의 목록:\n{queries_text}\n\n"
                    "각 질의에 대한 검색 결과를 정리해주세요."
                )
            ),
        ]
    )
    
    # 수집 및 정리된 텍스트를 retrieved_docs 상태 키에 저장합니다.
    return {"retrieved_docs": msg.content}


### 6-3. Analysis Tool Agent (`analysis_tool_agent`)
수집된 원시 정보(`retrieved_docs`)를 사용자의 원본 요청(`input`) 및 작업 계획(`plan`)과 대조하여, 핵심 패턴과 통찰(Insight)을 도출하는 분석 전담 노드입니다.
- 단순 정보 나열에 그치지 않고, 다음 단계인 `Writer Agent`가 논리적인 글을 작성할 수 있도록 분석된 뼈대와 시사점을 정리해 주는 교량 역할을 수행합니다.


In [ ]:
def analysis_tool_agent(state: State):
    """수집된 정보를 심층 분석하여 보고서 작성에 필요한 핵심 인사이트를 도출합니다."""
    # 원본 요청, 작업 계획, 검색된 정보를 모두 컨텍스트로 제공하여 분석을 요청합니다.
    msg = llm_client.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 데이터 분석 에이전트입니다. "
                    "검색된 정보를 분석하여 핵심 인사이트, 패턴, 통찰을 도출하고, "
                    "작성자가 보고서를 작성하는 데 활용할 수 있도록 정리하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"원본 요청: {state['input']}\n\n"
                    f"작업 계획:\n{state['plan']}\n\n"
                    f"검색된 정보:\n{state['retrieved_docs']}\n\n"
                    "이 정보를 분석하여 핵심 인사이트를 도출해주세요."
                )
            ),
        ]
    )
    
    # 도출된 분석 결과를 analysis 상태 키에 업데이트합니다.
    return {"analysis": msg.content}


### 6-4. Writer Agent (`writer_agent`)
분석 결과(`analysis`)와 작업 계획(`plan`)을 토대로 최종 결과물의 초안(`draft`)을 작성하는 노드입니다.
- **최초 작성 시**: `review_feedback`이 비어 있으므로 분석 결과만을 바탕으로 초안을 작성합니다.
- **재작성(Revision) 시**: `Reviewer Agent` 또는 `Human Approval Node`에서 반려되어 돌아온 경우, `review_feedback`에 담긴 구체적인 지적 사항을 프롬프트 하단에 추가하여 피드백이 반영된 개선본을 작성하도록 지시합니다.


In [ ]:
def writer_agent(state: State):
    """분석 결과를 바탕으로 초안을 작성하거나, 검토 피드백을 반영하여 결과물을 재작성합니다."""
    # 이전 검토 단계에서 넘어온 피드백과 현재 작성 회차를 조회합니다.
    feedback = state.get("review_feedback", "")
    revision_count = state.get("revision_count", 0)
    
    system_prompt = (
        "당신은 전문 작성 에이전트입니다. "
        "분석 결과를 바탕으로 사용자의 요청에 부합하는 명확하고 체계적인 결과물을 작성하세요."
    )
    
    # 기본 작성 컨텍스트(원본 요청, 작업 계획, 분석 결과)를 구성합니다.
    user_content = (
        f"원본 요청: {state['input']}\n\n"
        f"작업 계획:\n{state['plan']}\n\n"
        f"분석 결과:\n{state['analysis']}\n\n"
    )
    
    # 만약 이전 검토(Reviewer 또는 사용자)에서 반려되어 피드백이 존재하는 경우,
    # 프롬프트에 피드백 내용을 명시적으로 추가하여 개선을 강제합니다.
    if feedback and revision_count > 0:
        user_content += (
            f"\n[이전 초안에 대한 검토 피드백 - 반드시 반영하여 개선하세요]\n"
            f"{feedback}\n\n"
            f"위 피드백을 반영하여 개선된 결과물을 작성하세요. (재작성 {revision_count}회차)"
        )
        
    # LLM을 호출하여 보고서 초안(또는 수정본)을 생성합니다.
    msg = llm_client.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_content),
        ]
    )
    
    # 작성된 결과물을 draft 상태 키에 저장합니다.
    return {"draft": msg.content}


### 6-5. Reviewer Agent (`reviewer_agent`)
`Writer Agent`가 작성한 초안(`draft`)의 품질을 객관적으로 검증하는 내부 감사(Self-Reflection) 노드입니다.
- `reviewer_llm`을 사용하여 `ReviewResult` 스키마(`feedback`, `approved`) 형태로 응답을 받습니다.
- 초안이 원본 요청을 충실히 반영하고 논리적 결함이 없다면 `approved=True`를 반환하고, 부족한 부분이 있다면 `approved=False`와 함께 구체적인 수정 지침을 `feedback`에 담아 반환합니다.


In [ ]:
def reviewer_agent(state: State):
    """작성된 초안의 품질과 요구사항 충족 여부를 평가하고 승인 여부 및 피드백을 반환합니다."""
    # ReviewResult 스키마가 바인딩된 reviewer_llm을 호출하여 정형화된 평가 결과를 얻습니다.
    result = reviewer_llm.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 엄격한 검토자 에이전트입니다. "
                    "초안이 사용자의 원본 요청을 충족하는지, 내용이 정확하고 체계적인지 평가하세요. "
                    "명백한 결함이 없으면 approved=true로 설정하고, 개선이 필요하면 "
                    "approved=false와 함께 구체적인 피드백을 제공하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"원본 요청: {state['input']}\n\n"
                    f"초안:\n{state['draft']}\n\n"
                    "초안을 검토하고 승인 여부를 결정해주세요."
                )
            ),
        ]
    )
    
    # 평가 피드백과 승인 여부(Boolean)를 State에 업데이트합니다.
    return {
        "review_feedback": result.feedback,
        "review_approved": result.approved,
    }


### 6-6. Human Approval Node (`human_approval_node`)
내부 검토(`Reviewer Agent`)를 통과한 초안을 최종 사용자에게 제시하고 승인 여부를 입력받는 **Human-in-the-Loop (HITL)** 노드입니다.

- **`interrupt()`의 동작 원리**:
  1. 노드 실행 중 `interrupt(...)`를 만나면 LangGraph는 현재까지의 그래프 상태(`State`)를 체크포인터(`MemorySaver`)에 저장하고 실행을 즉시 일시 중지합니다.
  2. 외부 루프에서 사용자가 의사결정을 입력한 뒤 `Command(resume=user_decision)`으로 그래프를 재개하면, `human_approval_node` 함수가 처음부터 다시 실행되면서 `interrupt()` 호출부가 사용자의 입력값(`user_decision`)을 반환합니다.
- **분기 처리**:
  - 사용자가 `'approve'`를 입력한 경우: `human_approved: True`로 설정하고 현재 초안을 `final_output`으로 확정합니다.
  - 그 외의 텍스트(구체적인 수정 요청사항 등)를 입력한 경우: 이를 사용자의 반려 피드백으로 간주하여 `review_feedback`에 저장하고 `human_approved: False`로 설정합니다. 이후 라우터에 의해 `Writer Agent`로 되돌아가 사용자 피드백을 반영한 재작성을 수행합니다.


In [ ]:
def human_approval_node(state: State):
    """실행을 일시 중지(interrupt)하고 사용자로부터 최종 승인 또는 수정 피드백을 입력받습니다."""
    # 사용자가 검토할 수 있도록 원본 요청과 현재 작성된 초안을 화면에 출력합니다.
    print("\n" + "=" * 60)
    print("[Human Approval Node] 최종 결과물 검토 대기 중...")
    print("=" * 60)
    print(f"\n[원본 요청]\n{state['input']}")
    print(f"\n[작성된 결과물]\n{state['draft']}")
    print("\n" + "=" * 60)

    # interrupt()를 호출하여 그래프 실행을 멈추고 외부(사용자) 입력을 대기합니다.
    # 이후 Command(resume=값)가 호출되면 해당 값이 user_decision 변수에 할당됩니다.
    user_decision = interrupt(
        {
            "question": "이 결과물을 승인하시겠습니까? (승인 시 'approve', 반려 시 구체적인 수정 요청사항 입력)",
            "draft": state["draft"],
            # "options": ["approve", "reject"],
        }
    )
    
    # resume을 통해 전달된 값이 'approve'이면 최종 승인으로 처리합니다.
    if user_decision == "approve":
        print("\n[Human Approval] 사용자가 승인했습니다.")
        return {
            "human_approved": True,
            "final_output": state["draft"],
        }
    # 'approve'가 아닌 다른 텍스트가 입력된 경우 '반려 및 수정 요청 피드백'으로 간주합니다.
    else:
        # 빈 문자열이나 단순 'reject'만 입력된 경우를 대비하여 기본 피드백 문구를 지정합니다.
        feedback = (
            user_decision
            if isinstance(user_decision, str) and user_decision not in ["", "reject"]
            else "사용자가 결과물을 반려했습니다. 내용을 전반적으로 개선해주세요."
        )
        
        print(f"\n[Human Approval] 사용자가 반려했습니다. 피드백: {feedback}")
        return {
            "human_approved": False,
            "review_feedback": f"[사용자 최종 반려 피드백] {feedback}",  # Writer Agent에게 전달될 피드백
            "final_output": state["draft"],  # 최대 재작성 횟수(5회) 초과로 강제 종료될 때 마지막 초안을 보존하기 위함
        }


## 7. Supervisor 조건부 라우팅(Routing) 및 보조 노드 정의

워크플로우의 제어 흐름과 반복 횟수를 통제하는 라우팅 함수들과 카운터 노드를 정의합니다.

1. **`route_after_review(state)`**:
   - `Reviewer Agent` 실행 직후의 분기를 결정합니다.
   - 검토를 통과(`review_approved == True`)했거나, 이미 작성 횟수(`revision_count`)가 **3회 이상**에 도달한 경우에는 무한 루프를 막기 위해 `human_approval` 노드로 이동합니다.
   - 그렇지 않으면 `writer` 분기(실제 그래프에서는 `increment_revision` 노드)로 되돌려 초안을 재작성하게 합니다.
2. **`route_after_human(state)`**:
   - `Human Approval Node` 실행 직후의 분기를 결정합니다.
   - 사용자가 승인(`human_approved == True`)했거나, 총 누적 작성 횟수가 **5회 이상**에 도달한 경우 워크플로우를 최종 종료(`END`)합니다.
   - 사용자가 반려하고 아직 5회 미만인 경우, 다시 `writer` 분기(`increment_revision` 노드)로 이동하여 사용자의 피드백을 반영합니다.
3. **`increment_revision(state)`**:
   - `Writer Agent`가 실행되기 바로 전에 호출되어 `revision_count`를 1 증가시키는 보조 노드입니다.


In [ ]:
def route_after_review(state: State) -> Literal["human_approval", "writer"]:
    """Reviewer Agent의 검토 결과 및 작성 횟수를 확인하여 다음 이동할 노드를 결정합니다."""
    # 1) 검토자가 초안을 승인한 경우 사용자 최종 확인 단계로 이동합니다.
    if state.get("review_approved", False):
        return "human_approval"
    
    # 2) 검토를 통과하지 못했더라도 작성 횟수가 3회 이상이면 무한 루프 방지를 위해 강제로 사용자 확인 단계로 넘깁니다.
    if state.get("revision_count", 0) >= 3:
        print("\n[Supervisor] 최대 재작성 횟수(3회) 도달 → Human Approval로 강제 진행")
        return "human_approval"
    
    # 3) 검토 미통과 및 3회 미만인 경우 Writer Agent로 되돌려 재작성을 지시합니다.
    print(f"\n[Supervisor] 검토 미통과 → Writer Agent로 재작성 지시 ({state.get('revision_count', 0) + 1}회차)")
    return "writer"


def route_after_human(state: State) -> Literal[END, "writer"]:
    """사용자의 승인 여부 및 총 작성 횟수를 확인하여 종료(END) 또는 재작성(writer)을 결정합니다."""
    # 1) 사용자가 최종 승인한 경우 워크플로우를 즉시 종료(END)합니다.
    if state.get("human_approved", False):
        return END
    
    # 2) 사용자가 반려했더라도 총 누적 작성 횟수가 5회 이상이면 현재 결과물로 강제 종료합니다.
    revision_count = state.get("revision_count", 0)
    if revision_count >= 5:
        print("\n[Supervisor] 사용자 거부 횟수(5회) 초과 → 현재 결과물로 종료")
        return END
    
    # 3) 사용자가 반려하고 5회 미만인 경우 Writer Agent로 되돌려 재작성을 지시합니다.
    print(f"\n[Supervisor] 사용자 거부 → Writer Agent로 재작성 지시 ({revision_count + 1}회차)")
    return "writer"


def increment_revision(state: State):
    """Writer Agent가 호출되기 직전에 실행되어 작성 횟수(revision_count)를 1 증가시킵니다."""
    return {"revision_count": state.get("revision_count", 0) + 1}


## 8. 상태 그래프(StateGraph) 조립 및 컴파일

앞서 정의한 7개의 노드와 엣지(일반 엣지 및 조건부 엣지)를 `StateGraph(State)` 빌더에 등록하고 하나의 실행 가능한 워크플로우로 컴파일합니다.
- **주의할 점**: 조건부 엣지 매핑에서 `"writer": "increment_revision"`으로 연결하여, 재작성 루프가 발생할 때마다 반드시 `increment_revision` 노드를 먼저 거쳐 `revision_count`가 정확히 카운팅되도록 설계되어 있습니다.
- **`MemorySaver` 체크포인터**: `human_approval` 노드의 `interrupt()` 상태에서 그래프의 메모리 스냅샷을 보존하고 이후 `thread_id`를 기반으로 재개(`resume`)하기 위해 반드시 체크포인터를 함께 컴파일해야 합니다.


In [ ]:
# State 스키마를 기반으로 StateGraph 빌더를 초기화합니다.
supervisor_builder = StateGraph(State)

# 각 에이전트 및 유틸리티 함수를 그래프의 노드로 등록합니다.
supervisor_builder.add_node("request_planner", request_planner_agent)
supervisor_builder.add_node("retrieval", retrieval_agent)
supervisor_builder.add_node("analysis", analysis_tool_agent)
supervisor_builder.add_node("increment_revision", increment_revision)
supervisor_builder.add_node("writer", writer_agent)
supervisor_builder.add_node("reviewer", reviewer_agent)
supervisor_builder.add_node("human_approval", human_approval_node)

# 순차적으로 진행되는 기본 엣지(Edge)들을 연결합니다.
supervisor_builder.add_edge(START, "request_planner")
supervisor_builder.add_edge("request_planner", "retrieval")
supervisor_builder.add_edge("retrieval", "analysis")
supervisor_builder.add_edge("analysis", "increment_revision")
supervisor_builder.add_edge("increment_revision", "writer")
supervisor_builder.add_edge("writer", "reviewer")

# Reviewer 노드 이후의 조건부 엣지를 설정합니다.
# 'writer'가 반환되면 카운터 증가를 위해 'increment_revision' 노드로 이동시킵니다.
supervisor_builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "human_approval": "human_approval",
        "writer": "increment_revision",
    },
)

# Human Approval 노드 이후의 조건부 엣지를 설정합니다.
# 승인 시 END로 종료하고, 반려('writer') 시 'increment_revision' 노드로 이동시킵니다.
supervisor_builder.add_conditional_edges(
    "human_approval",
    route_after_human,
    {
        END: END,
        "writer": "increment_revision",
    },
)

# 인메모리 체크포인터(MemorySaver)를 생성하여 interrupt 시점의 상태를 저장할 수 있도록 컴파일합니다.
checkpointer = MemorySaver()
supervisor_workflow = supervisor_builder.compile(checkpointer=checkpointer)


## 9. 워크플로우 그래프 시각화

컴파일된 `supervisor_workflow`의 전체 노드 연결 구조와 조건부 분기 경로를 Mermaid 다이어그램 기반의 PNG 이미지로 렌더링하여 화면에 출력하고 로컬 파일(`Supervisor.png`)로 저장합니다.


In [ ]:
# 그래프 구조를 Mermaid PNG 바이트 데이터로 변환합니다.
png = supervisor_workflow.get_graph().draw_mermaid_png()

# 변환된 이미지를 현재 디렉터리에 'Supervisor.png' 파일로 저장합니다.
Path("Supervisor.png").write_bytes(png)

# Jupyter 노트북 출력 셀에 그래프 이미지를 표시합니다.
display(Image(data=png))
print("saved Supervisor.png")


## 10. 워크플로우 실행 및 대화형 인터럽트(HITL) 루프 제어

초기 상태(`initial_state`)와 스레드 식별자(`thread_id`)를 설정하여 워크플로우를 실행합니다.

### 실행 단계 설명
1. **최초 실행 (`invoke`)**:
   - `START`부터 시작하여 `request_planner` → `retrieval` → `analysis` → `increment_revision` → `writer` → `reviewer`까지 자동 실행됩니다.
   - 내부 검토를 통과(또는 3회 도달)하면 `human_approval` 노드로 진입하며, 내부의 `interrupt()` 호출에 의해 실행이 일시 중지됩니다.
2. **대화형 인터럽트 루프 (`while True`)**:
   - `supervisor_workflow.get_state(config)`를 통해 현재 그래프의 스냅샷을 조회합니다.
   - `state_snapshot.next`가 비어 있다면(`()`) 그래프가 `END`에 도달하여 완전히 종료된 것이므로 루프를 탈출합니다.
   - 대기 중인 노드가 있다면 `input()` 프롬프트를 띄워 사용자로부터 `'approve'` 또는 수정 요청사항을 입력받습니다.
   - 입력받은 값을 `Command(resume=user_decision)`에 담아 다시 `invoke()`를 호출함으로써 멈춰 있던 지점부터 그래프를 재개합니다.
3. **최종 결과 출력**:
   - 루프 종료 후 최종 상태값(`final_result`)에서 승인 여부, 총 재작성 횟수, 최종 완성된 보고서(`final_output`)를 출력합니다.


In [ ]:
# =========================================================
# 1. 초기 상태 설정 및 최초 실행
# =========================================================
# 사용자 요청(input)과 초기 작성 카운터(revision_count=0)를 정의합니다.
initial_state = {
    "input": "AI 에이전트의 발전 과정과 향후 전망에 대한 보고서를 작성해줘",
    "revision_count": 0,
}

print("워크플로우 최초 실행 시작...\n")
# 체크포인터가 실행 세션을 구분할 수 있도록 고유한 thread_id를 설정합니다.
config = {"configurable": {"thread_id": "supervisor-agent"}}

# 최초 실행을 시작합니다. (human_approval 노드의 interrupt 지점까지 자동으로 진행됩니다.)
result = supervisor_workflow.invoke(initial_state, config=config)


# =========================================================
# 2. 대화형 인터럽트 처리 루프 (Human-in-the-Loop)
# =========================================================
# 워크플로우가 최종 종료(END)될 때까지 사용자의 입력을 받아 재개하는 과정을 반복합니다.
while True:
    # 현재 thread_id에 저장된 그래프 상태 스냅샷을 조회합니다.
    state_snapshot = supervisor_workflow.get_state(config)
    
    # state_snapshot.next 값이 비어 있다면(빈 튜플), 모든 노드가 완료되어 END에 도달한 것입니다.
    if not state_snapshot.next:
        break
        
    # next 값이 존재한다면 human_approval_node의 interrupt()에서 멈춰 있는 상태입니다.
    current_values = state_snapshot.values
    print("\n" + "=" * 60)
    print(f"[초안 대기 중] 현재 draft 길이: {len(current_values.get('draft', ''))}자")
    print(f"[재작성 횟수]: {current_values.get('revision_count', 0)}회")
    print("=" * 60)
    
    # 사용자로부터 승인('approve') 또는 구체적인 반려/수정 사유를 입력받습니다.
    user_decision = input(
        "\n결과물을 승인하시겠습니까?\n"
        "(승인하려면 'approve' 입력, 반려하려면 구체적인 수정 요청사항 입력): "
    )
           
    print(f"\n입력된 내용: {user_decision}")
    print("워크플로우 재개 중...\n")
    
    # Command(resume=...)를 통해 사용자의 입력값을 interrupt() 호출 지점으로 전달하며 그래프를 재개합니다.
    supervisor_workflow.invoke(Command(resume=user_decision), config=config)


# =========================================================
# 3. 최종 결과 확인
# =========================================================
# 완료된 그래프의 최종 상태 딕셔너리를 가져옵니다.
final_result = supervisor_workflow.get_state(config).values

print("\n" + "=" * 60)
print("\n최종 완료!")
print(f"사용자 최종 승인 여부: {final_result.get('human_approved')}")
print(f"총 재작성 횟수: {final_result.get('revision_count')}")
print("\n[최종 결과물]")
print("=" * 60)
print(final_result.get("final_output"))
